In [5]:
import pandas as pd
import numpy as np
import sqlite3
import os
 
RAW_PATH = "stock_raw_combined.csv"
CLEAN_CSV_PATH = "stock_cleaned.csv"
SQLITE_DB_PATH = "stock_data.db"

### Function to extract the Data

In [6]:
def extract(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["Date"])
    print(f"Extracted {len(df)} rows from {path}")
    return df

### Transforming data

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Handle missing values, duplicates, and data types."""
    before = len(df)
    df = df.drop_duplicates(subset=["Date", "Ticker"])
    df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
 
    # Dropping rows with nulls in critical columns
    df = df.dropna(subset=["Open", "High", "Low", "Close", "Volume"])
    numeric_cols = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=numeric_cols)
 
    # Remove impossible values like negative/zero prices
    df = df[(df["Open"] > 0) & (df["High"] > 0) & (df["Low"] > 0) & (df["Close"] > 0)]
    print(f"Cleaning: {before} -> {len(df)} rows ({before - len(df)} removed)")
    return df

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add technical indicators & features, computed per-ticker."""
    out = []
    for ticker, g in df.groupby("Ticker"):
        g = g.sort_values("Date").copy()
        g["Daily_Return"] = g["Close"].pct_change() * 100
        g["MA_7"] = g["Close"].rolling(window=7).mean()
        g["MA_30"] = g["Close"].rolling(window=30).mean()
        g["Volatility_7"] = g["Daily_Return"].rolling(window=7).std()
        delta = g["Close"].diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)
        avg_gain = gain.rolling(14).mean()
        avg_loss = loss.rolling(14).mean()
        rs = avg_gain / avg_loss.replace(0, np.nan)
        g["RSI_14"] = 100 - (100 / (1 + rs))
 
        # Lag features (previous days' close - helps prediction)
        for lag in [1, 2, 3]:
            g[f"Close_Lag_{lag}"] = g["Close"].shift(lag)
        g["Target_Next_Return"] = ((g["Close"].shift(-1) - g["Close"])/ g["Close"]) * 100
        out.append(g)
 
    result = pd.concat(out, ignore_index=True)
    result = result.dropna().reset_index(drop=True)
    print(f"Feature engineering complete: {result.shape[1]} columns, {len(result)} rows after dropping NaNs")
    return result
 
def transform(df: pd.DataFrame) -> pd.DataFrame:
    df = clean_data(df)
    df = engineer_features(df)
    return df

### Loading the data

In [8]:
def load(df: pd.DataFrame, csv_path: str, db_path: str):
    df.to_csv(csv_path, index=False)
    print(f"Saved cleaned CSV -> {csv_path}")
 
    conn = sqlite3.connect(db_path)
    df.to_sql("stock_features", conn, if_exists="replace", index=False)
    conn.close()
    print(f"Loaded into SQLite -> {db_path} (table: stock_features)")
 
def main():
    df = extract(RAW_PATH)
    df = transform(df)
    load(df, CLEAN_CSV_PATH, SQLITE_DB_PATH)

if __name__ == "__main__":
    main()

Extracted 7576 rows from stock_raw_combined.csv
Cleaning: 7576 -> 7576 rows (0 removed)
Feature engineering complete: 17 columns, 7456 rows after dropping NaNs
Saved cleaned CSV -> stock_cleaned.csv
Loaded into SQLite -> stock_data.db (table: stock_features)
